# Python Descriptors — Advanced Problems with Fully Worked Solutions

**Audience:** Intermediate-to-advanced Python developers. **Runtime:** Python 3.10+; Python standard library only. **Run order:** Execute cells top to bottom. All examples use independent class names to avoid accidental redefinitions. **Verification:** Every solution includes executable `assert` statements, often including failure-path tests.

This notebook extends the introductory `TimeUTC`, `Choice`, `Deck`, and `Dice` examples into descriptor internals and production-quality patterns. A descriptor is an object assigned to a *class attribute* that defines `__get__`, `__set__`, and/or `__delete__`; `__set_name__` supplies its defining owner and attribute name at class creation.

### Learning outcomes

- Predict attribute lookup and distinguish data from non-data descriptors.
- Use `__set_name__` correctly across inheritance and dynamic class modifications.
- Build reusable validated, read-only, write-once, and auditable fields.
- Implement lazy evaluation for dictionary-backed and slotted instances, including weak-reference caveats.
- Understand function binding, `property`, `classmethod`, `staticmethod`, and `inspect.getattr_static`.
- Design atomic multi-field updates and a schema-driven descriptor model.
- Explain thread safety, introspection, performance, and the limits of descriptor-based enforcement.

**Exercise pattern:** Read each challenge and attempt it before reading the solution. Tests are included directly below each solution, with no third-party test framework required.

## Essential reference: the lookup rules

For ordinary `obj.name` lookup (assuming no custom `__getattribute__`), Python searches the class MRO for the attribute and broadly applies this precedence:

1. A **data descriptor** on the class/MRO (`__set__` **or** `__delete__` is defined).
2. The instance dictionary, if the instance has one.
3. A **non-data descriptor** on the class/MRO (`__get__` without `__set__` and `__delete__`), or an ordinary class attribute.
4. `__getattr__`, if ordinary lookup raises `AttributeError`.

For class access, well-behaved descriptors commonly implement `if instance is None: return self` in `__get__`. That is a useful *convention*, not a protocol requirement. Assigning a value directly to `obj.__dict__` can bypass validation during the assignment itself; a data descriptor still takes precedence when the same public name is read.

A descriptor implementing `__delete__` but not `__set__` is **still a data descriptor**. `__set_name__` is invoked automatically when the containing class is *created*, not when descriptors are assigned to an existing class afterward.

The original example used `datetime.utcnow()`, which returns a naive datetime and is deprecated in modern Python. For UTC values use `datetime.now(timezone.utc)`; return the descriptor for class-level inspection.

In [1]:
from __future__ import annotations

from dataclasses import InitVar, dataclass, fields
from datetime import datetime, timezone
from functools import wraps
from inspect import getattr_static, isdatadescriptor
from random import Random
from threading import Lock, Thread
from types import MethodType
from weakref import WeakKeyDictionary
import gc
import json
import math
import weakref

# Reusable test helper. Keep expected exceptions explicit, never blanket-except.
def expect_raises(exception_type, action):
    try:
        action()
    except exception_type as exc:
        return exc
    except Exception as exc:
        raise AssertionError(
            f"Expected {exception_type.__name__}, got {type(exc).__name__}: {exc}"
        ) from exc
    raise AssertionError(f"Expected {exception_type.__name__} to be raised")

print("Ready: standard-library imports and assertion helper loaded.")

Ready: standard-library imports and assertion helper loaded.


## Problem 01 — Fix the introductory clock and random-choice descriptors

**Challenge.** Create a timezone-aware UTC descriptor and a random-choice descriptor that supports a seeded, *local* random-number generator, validates an empty choice set, and returns itself on class access. Demonstrate both from two unrelated owner classes.

**Design hints.** Keep random state in an explicit `Random` instance rather than reseeding the global generator. Use `datetime.now(timezone.utc)`.

### Solution 01

**Reference implementation and executable tests**

In [2]:
class UTCClock:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return datetime.now(timezone.utc).isoformat()


class RandomChoice:
    def __init__(self, *options, rng=None):
        if not options:
            raise ValueError("At least one option is required")
        self._options = tuple(options)
        self._rng = rng if rng is not None else Random()

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return self._rng.choice(self._options)


class EventLog:
    timestamp = UTCClock()


class CardSampler:
    rank = RandomChoice(*"23456789JQKA", "10", rng=Random(7))
    suit = RandomChoice("Spades", "Hearts", "Diamonds", "Clubs", rng=Random(8))


assert isinstance(EventLog.timestamp, UTCClock)
assert datetime.fromisoformat(EventLog().timestamp).tzinfo is not None
assert isinstance(CardSampler.rank, RandomChoice)
assert CardSampler().rank in set("23456789JQKA") | {"10"}
assert CardSampler().suit in {"Spades", "Hearts", "Diamonds", "Clubs"}
expect_raises(ValueError, lambda: RandomChoice())
print("Problem 01 passed: UTC and independently seeded choice streams.")

Problem 01 passed: UTC and independently seeded choice streams.


**Why it works / design review.** The clock computes afresh on every instance access. The random-choice descriptor is reusable but its injected generator is shared by all instances using that *particular descriptor*. Separate generators if independent streams are required.

## Problem 02 — Prove attribute precedence by experiment

**Challenge.** Create both descriptor categories under the *same* owner. Inject instance-dictionary shadow entries for both attributes, then verify which descriptor wins. Also verify what direct class dictionary lookup returns.

**Design hints.** Data descriptors must provide `__set__` or `__delete__`; a plain `__get__` makes a non-data descriptor. Inspect `vars(instance)` and `vars(Owner)` separately.

### Solution 02

**Reference implementation and executable tests**

In [3]:
class NonDataMarker:
    def __get__(self, instance, owner):
        return self if instance is None else "non-data descriptor"


class DataMarker:
    def __get__(self, instance, owner):
        return self if instance is None else "data descriptor"

    def __set__(self, instance, value):
        instance.__dict__["_data_value"] = value


class PrecedenceCase:
    non_data = NonDataMarker()
    data = DataMarker()


case = PrecedenceCase()
assert case.non_data == "non-data descriptor"
assert case.data == "data descriptor"
case.__dict__["non_data"] = "instance wins"
case.__dict__["data"] = "instance loses"
assert case.non_data == "instance wins"
assert case.data == "data descriptor"
case.data = 42
assert case.__dict__["_data_value"] == 42
assert isinstance(vars(PrecedenceCase)["data"], DataMarker)
assert PrecedenceCase.non_data is vars(PrecedenceCase)["non_data"]
assert PrecedenceCase.data is vars(PrecedenceCase)["data"]
print("Problem 02 passed:", vars(case))

Problem 02 passed: {'non_data': 'instance wins', 'data': 'instance loses', '_data_value': 42}


**Why it works / design review.** An instance dictionary entry shadows a non-data descriptor but not a data descriptor. A direct `vars(Owner)["name"]` lookup retrieves the underlying descriptor without invoking `__get__`.

## Problem 03 — Build a computed read-only data descriptor

**Challenge.** Implement a `slug` computed from an instance's `title`. Prevent ordinary assignment and deletion. Show that even inserting a fake `slug` key into the instance dictionary does not change the observed result.

**Design hints.** Implement `__set__` and `__delete__` to raise `AttributeError`. Access the owner-level descriptor via the class.

### Solution 03

**Reference implementation and executable tests**

In [4]:
class Slug:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return "-".join(instance.title.casefold().split())

    def __set__(self, instance, value):
        raise AttributeError("slug is read-only")

    def __delete__(self, instance):
        raise AttributeError("slug cannot be deleted")


class Article:
    slug = Slug()

    def __init__(self, title):
        self.title = title


article = Article("  Python   Descriptors  ")
assert article.slug == "python-descriptors"
assert Article.slug is vars(Article)["slug"]
article.__dict__["slug"] = "forged"
assert article.slug == "python-descriptors"
article.title = "Another Title"
assert article.slug == "another-title"
expect_raises(AttributeError, lambda: setattr(article, "slug", "tampered"))
expect_raises(AttributeError, lambda: delattr(article, "slug"))
print("Problem 03 passed: computed, read-only data descriptor.")

Problem 03 passed: computed, read-only data descriptor.


**Why it works / design review.** A read-only descriptor needs a raising `__set__` (or `__delete__`) to keep its public name ahead of the instance dictionary. Do not equate that with a security boundary: arbitrary mutation of underlying instance state remains possible.

## Problem 04 — Classify a delete-only descriptor correctly

**Challenge.** Write a descriptor that supports reads and deletion but not explicit setting. Prove that it is a *data* descriptor, wins over a forged instance attribute, and calls `__delete__` when deleted.

**Design hints.** Use a separate private storage key; implement `__delete__` without `__set__`. Inspect it using `inspect.isdatadescriptor`.

### Solution 04

**Reference implementation and executable tests**

In [5]:
class DeleteOnly:
    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get("_resource", "not configured")

    def __delete__(self, instance):
        instance.__dict__.pop("_resource", None)


class ResourceHolder:
    resource = DeleteOnly()


holder = ResourceHolder()
holder.__dict__["_resource"] = "socket"
holder.__dict__["resource"] = "forged"
assert holder.resource == "socket"
assert isdatadescriptor(ResourceHolder.resource)
expect_raises(AttributeError, lambda: setattr(holder, "resource", "replacement"))
del holder.resource
assert holder.resource == "not configured"
assert holder.__dict__["resource"] == "forged"
print("Problem 04 passed: delete-only is a data descriptor.")

Problem 04 passed: delete-only is a data descriptor.


**Why it works / design review.** Both `__set__` and `__delete__` individually cause data-descriptor precedence. Assignment to the public name still fails because this descriptor provides no `__set__`.

## Problem 05 — Capture names; handle dynamically installed descriptors

**Challenge.** Implement a named string field that uses `__set_name__` to derive a private storage key. Install one field inside a class statement and another afterward with `setattr`. Repair the second field correctly. Observe that an inherited descriptor's original owner registration does not change.

**Design hints.** Python does not automatically call `__set_name__` on descriptors assigned after class creation; call the method manually *after* `setattr`. Do not reuse one stateful descriptor under several names.

### Solution 05

**Reference implementation and executable tests**

In [6]:
class NamedText:
    def __set_name__(self, owner, name):
        self.defining_owner = owner
        self.public_name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.storage_name]

    def __set__(self, instance, value):
        if not isinstance(value, str):
            raise TypeError(f"{self.public_name} must be a string")
        instance.__dict__[self.storage_name] = value


class Profile:
    first = NamedText()


class PremiumProfile(Profile):
    pass


assert Profile.first.public_name == "first"
assert PremiumProfile.first.defining_owner is Profile
last_descriptor = NamedText()
setattr(Profile, "last", last_descriptor)
assert not hasattr(last_descriptor, "storage_name")
last_descriptor.__set_name__(Profile, "last")
profile = PremiumProfile()
profile.first = "Ada"
profile.last = "Lovelace"
assert (profile.first, profile.last) == ("Ada", "Lovelace")
assert profile.__dict__ == {"_first": "Ada", "_last": "Lovelace"}
expect_raises(TypeError, lambda: setattr(profile, "last", 99))
print("Problem 05 passed: automatic, inherited and manual name registration.")

Problem 05 passed: automatic, inherited and manual name registration.


**Why it works / design review.** `__set_name__` receives the *defining* owner, not every subclass that inherits the attribute. Descriptor reuse across different names can overwrite stored metadata, so create separate descriptor instances or implement an explicit per-owner/name mapping.

## Problem 06 — Implement a strict, transactional bounded integer

**Challenge.** Write a reusable field enforcing an inclusive integer range. Reject `bool`, despite `isinstance(True, int)` being true. Reject invalid values *before* altering stored data and give descriptive error messages.

**Design hints.** Validate constructor bounds and assignment separately. Store values on the instance; never store an instance's field value on the shared descriptor.

### Solution 06

**Reference implementation and executable tests**

In [7]:
class BoundedInt:
    def __init__(self, minimum, maximum):
        if type(minimum) is not int or type(maximum) is not int or minimum > maximum:
            raise ValueError("Bounds must be integers in increasing order")
        self.minimum, self.maximum = minimum, maximum

    def __set_name__(self, owner, name):
        self.name = name
        self.key = f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.key]
        except KeyError:
            raise AttributeError(f"{self.name} has not been initialized") from None

    def __set__(self, instance, value):
        if type(value) is not int:
            raise TypeError(f"{self.name} must be an int, not {type(value).__name__}")
        if not self.minimum <= value <= self.maximum:
            raise ValueError(f"{self.name} must be between {self.minimum} and {self.maximum}")
        instance.__dict__[self.key] = value


class ExamResult:
    score = BoundedInt(0, 100)
    attempts = BoundedInt(1, 5)

    def __init__(self, score, attempts):
        self.score = score
        self.attempts = attempts


result = ExamResult(90, 2)
expect_raises(ValueError, lambda: setattr(result, "score", 101))
expect_raises(TypeError, lambda: setattr(result, "score", True))
expect_raises(ValueError, lambda: setattr(result, "attempts", 0))
assert (result.score, result.attempts) == (90, 2)
other_result = ExamResult(40, 1)
assert other_result.score == 40 and result.score == 90
expect_raises(ValueError, lambda: BoundedInt(5, 1))
expect_raises(AttributeError, lambda: ExamResult.__new__(ExamResult).score)
print("Problem 06 passed: validated values and rollback on failure.")

Problem 06 passed: validated values and rollback on failure.


**Why it works / design review.** Store state in each instance under a private name; an invalid assignment raises without changing the old value. `type(value) is int` deliberately excludes booleans and integer subclasses (a documented strict typing choice).

## Problem 07 — Compose validators and avoid mutable shared defaults

**Challenge.** Create a `Field` with callable validators and an optional `default_factory`. Ensure fresh lists for each instance and run every validator before committing an assigned value. Define a concrete task model.

**Design hints.** A default factory should run once per missing instance, not once when creating the descriptor. Do not use `[]` as a descriptor-level shared default.

### Solution 07

**Reference implementation and executable tests**

In [8]:
_MISSING = object()


def nonempty_text(value):
    if not isinstance(value, str) or not value.strip():
        raise ValueError("expected nonempty text")
    return value.strip()


def copy_string_list(value):
    if not isinstance(value, list) or any(not isinstance(item, str) for item in value):
        raise TypeError("expected a list of strings")
    return list(value)


class ComposedField:
    def __init__(self, *validators, default_factory=None):
        self.validators = validators
        self.default_factory = default_factory

    def __set_name__(self, owner, name):
        self.name, self.key = name, f"_{name}"

    def normalize(self, value):
        for validator in self.validators:
            value = validator(value)
        return value

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.key not in instance.__dict__:
            if self.default_factory is None:
                raise AttributeError(f"{self.name} has no value")
            self.__set__(instance, self.default_factory())
        return instance.__dict__[self.key]

    def __set__(self, instance, value):
        normalized = self.normalize(value)
        instance.__dict__[self.key] = normalized


class WorkTask:
    title = ComposedField(nonempty_text)
    labels = ComposedField(copy_string_list, default_factory=list)

    def __init__(self, title):
        self.title = title


first_task = WorkTask("  Review PR  ")
second_task = WorkTask("Write tests")
assert first_task.title == "Review PR"
assert first_task.labels is not second_task.labels
first_task.labels.append("urgent")
assert second_task.labels == []
caller_labels = ["docs"]
first_task.labels = caller_labels
caller_labels.append("not copied")
assert first_task.labels == ["docs"]
expect_raises(TypeError, lambda: setattr(first_task, "labels", [1]))
assert first_task.labels == ["docs"]
print("Problem 07 passed: composable validation, copying and independent defaults.")

Problem 07 passed: composable validation, copying and independent defaults.


**Why it works / design review.** Factories avoid mutable-default aliasing. The example returns a *copy* when accepting a list, so later mutation of the caller's original list does not mutate model state. Validation of later in-place changes to the returned list is **not** automatic; use an immutable value or a custom collection if required.

## Problem 08 — Keep cross-field invariants with an atomic update

**Challenge.** Represent an interval with `start <= end`. Ordinary assignments must enforce the invariant. Add `replace(start, end)` that can atomically change both bounds (even when no individual assignment order would work) and leaves the old pair unchanged if validation fails.

**Design hints.** Single-field validation belongs in the descriptor; coordinated updates belong in the owner model. Stage *both* inputs before changing either stored value.

### Solution 08

**Reference implementation and executable tests**

In [9]:
class IntervalBoundary:
    def __set_name__(self, owner, name):
        self.name, self.key = name, f"_{name}"
        self.other_key = "_end" if name == "start" else "_start"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.key]

    def __set__(self, instance, value):
        if type(value) not in (int, float) or not math.isfinite(value):
            raise TypeError("bounds must be finite real numbers, excluding bool")
        other = instance.__dict__.get(self.other_key)
        if other is not None:
            if self.name == "start" and value > other:
                raise ValueError("start cannot exceed end")
            if self.name == "end" and value < other:
                raise ValueError("end cannot precede start")
        instance.__dict__[self.key] = value


class Interval:
    start = IntervalBoundary()
    end = IntervalBoundary()

    def __init__(self, start, end):
        self.replace(start, end)

    def replace(self, start, end):
        for value in (start, end):
            if type(value) not in (int, float) or not math.isfinite(value):
                raise TypeError("bounds must be finite real numbers")
        if start > end:
            raise ValueError("start cannot exceed end")
        # The pair is validated before either private value is committed.
        self.__dict__.update({"_start": start, "_end": end})


interval = Interval(1, 3)
expect_raises(ValueError, lambda: setattr(interval, "start", 4))
expect_raises(ValueError, lambda: interval.replace(10, 9))
assert (interval.start, interval.end) == (1, 3)
interval.replace(10, 12)
assert (interval.start, interval.end) == (10, 12)
expect_raises(TypeError, lambda: interval.replace(float("nan"), 12))
assert (interval.start, interval.end) == (10, 12)
print("Problem 08 passed: coordinated changes preserve interval invariants.")

Problem 08 passed: coordinated changes preserve interval invariants.


**Why it works / design review.** A model-level transaction is preferable to having unrelated descriptors manage partial multi-field updates. Direct writes to private storage are contained inside one audited model method *after* validation; a more elaborate persistence model would need its own transaction mechanism.

## Problem 09 — Write a lazy, non-data cached property

**Challenge.** Implement a decorator-style descriptor that computes once per instance, caches under the *public* attribute name, and permits replacement or invalidation by modifying/deleting that instance attribute. Verify independent caches.

**Design hints.** A non-data descriptor loses to a same-named entry in the instance dictionary. Use `__set_name__` and avoid storing computed values on the descriptor.

### Solution 09

**Reference implementation and executable tests**

In [10]:
class LazyCached:
    def __init__(self, function):
        self.function = function
        wraps(function)(self)

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        value = self.function(instance)
        instance.__dict__[self.name] = value
        return value


class Report:
    def __init__(self, base):
        self.base = base
        self.calls = 0

    @LazyCached
    def summary(self):
        self.calls += 1
        return self.base * 10


report_a, report_b = Report(3), Report(5)
assert isinstance(Report.summary, LazyCached)
assert (report_a.summary, report_a.summary, report_a.calls) == (30, 30, 1)
assert (report_b.summary, report_b.calls) == (50, 1)
report_a.summary = 999
assert report_a.summary == 999 and report_a.calls == 1
del report_a.summary
assert report_a.summary == 30 and report_a.calls == 2
print("Problem 09 passed: lazy evaluation, shadowing and invalidation.")

Problem 09 passed: lazy evaluation, shadowing and invalidation.


**Why it works / design review.** This is the fundamental strategy of `functools.cached_property`: cache in a per-instance dictionary and let it shadow the non-data descriptor. It requires a mutable instance `__dict__`; it is not automatically thread-safe.

## Problem 10 — Add a failure-safe cached property

**Challenge.** Modify the caching idea so a computation that raises does not leave a cached value; a later read may retry successfully. Show exactly two calls: one failure and one success.

**Design hints.** Never insert a cache entry until the computation has returned successfully. A plain non-data descriptor can already implement this.

### Solution 10

**Reference implementation and executable tests**

In [11]:
class FailureSafeCached:
    def __init__(self, function):
        self.function = function

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        result = self.function(instance)  # may raise; dictionary remains untouched
        instance.__dict__[self.name] = result
        return result


class FlakyService:
    def __init__(self):
        self.calls = 0

    @FailureSafeCached
    def token(self):
        self.calls += 1
        if self.calls == 1:
            raise RuntimeError("temporary outage")
        return "ready"


service = FlakyService()
expect_raises(RuntimeError, lambda: service.token)
assert "token" not in vars(service)
assert service.token == "ready"
assert service.token == "ready"
assert service.calls == 2
print("Problem 10 passed: failures do not poison the cache.")

Problem 10 passed: failures do not poison the cache.


**Why it works / design review.** Because assignment to the cache happens only after the function returns, exceptions do not poison the cached property. Choose retry behavior deliberately: some applications instead cache exception objects.

## Problem 11 — Cache values on slotted instances with weak keys

**Challenge.** Implement a descriptor using `WeakKeyDictionary` for objects that have `__slots__` but no instance dictionary. Make invalidation explicit and show class-level descriptor introspection.

**Design hints.** Slot instances must support weak references (`__weakref__` in slots unless inherited) *and* must be hashable for `WeakKeyDictionary`. Document that cached values must not strongly retain their own instance.

### Solution 11

**Reference implementation and executable tests**

In [12]:
class WeakCached:
    def __init__(self, function):
        self.function = function
        self.cache = WeakKeyDictionary()

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if instance not in self.cache:
            self.cache[instance] = self.function(instance)
        return self.cache[instance]

    def invalidate(self, instance):
        self.cache.pop(instance, None)


class SlottedCounter:
    __slots__ = ("number", "calls", "__weakref__")

    def __init__(self, number):
        self.number, self.calls = number, 0

    @WeakCached
    def doubled(self):
        self.calls += 1
        return self.number * 2


slot_a, slot_b = SlottedCounter(10), SlottedCounter(7)
assert slot_a.doubled == slot_a.doubled == 20
assert (slot_a.calls, slot_b.doubled, slot_b.calls) == (1, 14, 1)
assert len(SlottedCounter.doubled.cache) == 2
SlottedCounter.doubled.invalidate(slot_a)
assert slot_a.doubled == 20 and slot_a.calls == 2
expect_raises(TypeError, lambda: vars(slot_a))
print("Problem 11 passed: weak-key cache for weakrefable slotted owners.")

Problem 11 passed: weak-key cache for weakrefable slotted owners.


**Why it works / design review.** Weak keys do not keep their owners alive by themselves. However, if a cached **value** refers strongly to its key, that reference can keep the owner alive: weak-key caching is not a universal memory-leak cure. `WeakKeyDictionary` is not an automatic thread-safety primitive.

## Problem 12 — Handle unhashable slotted owners without weak dictionaries

**Challenge.** Demonstrate why a weak-key cache cannot be used for an unhashable slotted instance, then implement a descriptor whose cache is stored in a dedicated private slot. Include deletion to invalidate the cache.

**Design hints.** Use an unhashable `__hash__ = None` class to exhibit the limitation. For the alternative, allocate the private cache slot in the owner and initialize it to a sentinel.

### Solution 12

**Reference implementation and executable tests**

In [13]:
class UnhashableSlotted:
    __slots__ = ("value", "__weakref__")
    __hash__ = None

    def __init__(self, value):
        self.value = value


weak_map = WeakKeyDictionary()
expect_raises(TypeError, lambda: weak_map.__setitem__(UnhashableSlotted(2), 4))

_SLOT_EMPTY = object()


class SlotStoredCache:
    def __init__(self, function):
        self.function = function

    def __set_name__(self, owner, name):
        self.slot = f"_{name}_cache"
        if self.slot not in vars(owner).get("__slots__", ()):
            raise TypeError(f"{owner.__name__} must declare slot {self.slot!r}")

    def __get__(self, instance, owner):
        if instance is None:
            return self
        value = getattr(instance, self.slot)
        if value is _SLOT_EMPTY:
            value = self.function(instance)
            setattr(instance, self.slot, value)
        return value

    def __delete__(self, instance):
        setattr(instance, self.slot, _SLOT_EMPTY)


class SlotOnlyReport:
    __slots__ = ("n", "calls", "_answer_cache")
    __hash__ = None

    def __init__(self, n):
        self.n, self.calls = n, 0
        self._answer_cache = _SLOT_EMPTY

    @SlotStoredCache
    def answer(self):
        self.calls += 1
        return self.n ** 2


slot_report = SlotOnlyReport(6)
assert slot_report.answer == slot_report.answer == 36
assert slot_report.calls == 1
del slot_report.answer
assert slot_report.answer == 36 and slot_report.calls == 2
expect_raises(AttributeError, lambda: setattr(slot_report, "answer", 999))
print("Problem 12 passed: unhashable slots and explicit slot-backed caching.")

Problem 12 passed: unhashable slots and explicit slot-backed caching.


**Why it works / design review.** Dedicated slots trade reusable, external weak-key storage for explicit per-owner storage. The cache descriptor below is a **data descriptor** because it defines `__delete__`; it therefore cannot be overridden by an instance-dictionary entry.

## Problem 13 — Prove what happens across inheritance and overrides

**Challenge.** Record the defining owner in `__set_name__` and the lookup owner passed to `__get__`. Read an inherited descriptor through a base class, a subclass, and a subclass instance. Then override a descriptor in another subclass.

**Design hints.** Use a class dictionary lookup to inspect the descriptor itself. Do not assume inherited descriptors get `__set_name__` called a second time.

### Solution 13

**Reference implementation and executable tests**

In [14]:
class OwnershipProbe:
    def __set_name__(self, owner, name):
        self.declared_in, self.name = owner, name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return self.declared_in.__name__, owner.__name__, self.name


class BaseRecord:
    field = OwnershipProbe()


class ChildRecord(BaseRecord):
    pass


class OverriddenRecord(BaseRecord):
    field = OwnershipProbe()


assert BaseRecord.field is ChildRecord.field
assert BaseRecord.field.declared_in is BaseRecord
assert ChildRecord().field == ("BaseRecord", "ChildRecord", "field")
assert BaseRecord().field == ("BaseRecord", "BaseRecord", "field")
assert OverriddenRecord.field is not BaseRecord.field
assert OverriddenRecord.field.declared_in is OverriddenRecord
assert OverriddenRecord().field == ("OverriddenRecord", "OverriddenRecord", "field")
print("Problem 13 passed: inheritance and override metadata.")

Problem 13 passed: inheritance and override metadata.


**Why it works / design review.** The `owner` argument to ordinary class/instance lookup identifies the lookup class (for these examples). `__set_name__` is tied to where the descriptor was originally declared. An overriding descriptor in a subclass is a separate object and receives its own `__set_name__` call.

## Problem 14 — Trace __getattribute__, __getattr__, and static inspection

**Challenge.** Intercept normal instance lookups without breaking Python's descriptor behavior; prove that a descriptor runs through `object.__getattribute__`, but `inspect.getattr_static` obtains the underlying descriptor without invoking it. Show when `__getattr__` runs.

**Design hints.** Delegate using `object.__getattribute__`. `__getattr__` is a fallback when ordinary lookup raises `AttributeError`; it is not a replacement for `__getattribute__`.

### Solution 14

**Reference implementation and executable tests**

In [15]:
class CountingDescriptor:
    def __init__(self):
        self.reads = 0

    def __get__(self, instance, owner):
        if instance is None:
            return self
        self.reads += 1
        return "bound-result"


class LookupTrace:
    item = CountingDescriptor()

    def __init__(self):
        self.lookups = []

    def __getattribute__(self, name):
        if name not in {"lookups", "__dict__", "__class__"}:
            object.__getattribute__(self, "lookups").append(name)
        return object.__getattribute__(self, name)

    def __getattr__(self, name):
        return f"fallback:{name}"


trace = LookupTrace()
raw = getattr_static(trace, "item")
assert raw is LookupTrace.item
assert raw.reads == 0
assert object.__getattribute__(trace, "item") == "bound-result"
assert raw.reads == 1
assert trace.item == "bound-result"
assert raw.reads == 2
assert trace.unknown == "fallback:unknown"
expect_raises(AttributeError, lambda: object.__getattribute__(trace, "unknown"))
assert trace.lookups.count("item") == 1
print("Problem 14 passed: custom lookup, fallback and no-bind inspection.")

Problem 14 passed: custom lookup, fallback and no-bind inspection.


**Why it works / design review.** Overriding `__getattribute__` incorrectly can bypass descriptors or cause infinite recursion. `getattr_static` is for inspection and intentionally avoids normal binding/dynamic lookup.

## Problem 15 — Recreate function binding and inspect built-in descriptors

**Challenge.** Build a non-data descriptor which turns a plain function into a bound method on instance access. Inspect an ordinary Python function, `property`, `staticmethod`, and `classmethod` through the class dictionary, and classify the relevant data-descriptor behavior.

**Design hints.** Use `types.MethodType` instead of manually inventing `self` injection. Access `vars(Owner)` to get raw wrapper objects before their `__get__` is invoked.

### Solution 15

**Reference implementation and executable tests**

In [16]:
class BindFunction:
    def __init__(self, function):
        self.function = function

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return MethodType(self.function, instance)


class BindingExamples:
    @BindFunction
    def multiply(self, factor):
        return self.base * factor

    def ordinary(self):
        return self.base

    @property
    def squared(self):
        return self.base ** 2

    @staticmethod
    def identity(value):
        return value

    @classmethod
    def class_name(cls):
        return cls.__name__

    def __init__(self, base):
        self.base = base


binding = BindingExamples(7)
assert binding.multiply(3) == 21
assert binding.multiply.__self__ is binding
assert BindingExamples.multiply is vars(BindingExamples)["multiply"]
assert binding.ordinary() == 7
assert BindingExamples.identity(5) == 5
assert BindingExamples.class_name() == "BindingExamples"
assert isdatadescriptor(vars(BindingExamples)["squared"])
assert not isdatadescriptor(vars(BindingExamples)["ordinary"])
assert binding.squared == 49
expect_raises(AttributeError, lambda: setattr(binding, "squared", 2))
print("Problem 15 passed: bound methods and built-in descriptor categories.")

Problem 15 passed: bound methods and built-in descriptor categories.


**Why it works / design review.** Ordinary functions bind because function objects implement the descriptor protocol. `property` is a data descriptor even if it has no setter. Functions and `staticmethod`/`classmethod` wrappers do not provide instance-level data-descriptor precedence in the way `property` does.

## Problem 16 — Write once, audit changes, and avoid storing secrets in logs

**Challenge.** Implement two independent descriptors: a write-once identifier and an auditable field. Audit *only successful* mutations, recording the field name and event type rather than the potentially sensitive field value. Support audited deletion.

**Design hints.** Store values per instance. Validate first; append an audit event only after the mutation succeeds. Make the write-once field a data descriptor.

### Solution 16

**Reference implementation and executable tests**

In [17]:
class WriteOnce:
    def __set_name__(self, owner, name):
        self.name, self.key = name, f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.key]
        except KeyError:
            raise AttributeError(f"{self.name} is not set") from None

    def __set__(self, instance, value):
        if self.key in instance.__dict__:
            raise AttributeError(f"{self.name} may only be assigned once")
        instance.__dict__[self.key] = value

    def __delete__(self, instance):
        raise AttributeError(f"{self.name} cannot be deleted")


class AuditedString:
    def __set_name__(self, owner, name):
        self.name, self.key = name, f"_{name}"

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__[self.key]

    def __set__(self, instance, value):
        if not isinstance(value, str) or not value:
            raise ValueError("value must be a nonempty string")
        event = "update" if self.key in instance.__dict__ else "create"
        instance.__dict__[self.key] = value
        instance.audit_events.append((self.name, event))

    def __delete__(self, instance):
        if self.key not in instance.__dict__:
            raise AttributeError(f"{self.name} is not set")
        del instance.__dict__[self.key]
        instance.audit_events.append((self.name, "delete"))


class Account:
    account_id = WriteOnce()
    secret = AuditedString()

    def __init__(self, account_id, secret):
        self.audit_events = []
        self.account_id = account_id
        self.secret = secret


account = Account("A-001", "first-secret")
expect_raises(AttributeError, lambda: setattr(account, "account_id", "A-002"))
expect_raises(ValueError, lambda: setattr(account, "secret", ""))
assert account.secret == "first-secret"
account.secret = "new-secret"
del account.secret
assert account.audit_events == [
    ("secret", "create"), ("secret", "update"), ("secret", "delete")
]
assert "first-secret" not in repr(account.audit_events)
expect_raises(AttributeError, lambda: delattr(account, "account_id"))
print("Problem 16 passed: write-once and mutation-only event audit.")

Problem 16 passed: write-once and mutation-only event audit.


**Why it works / design review.** Logging field names and event types avoids exposing arbitrary user input in the audit trail. This is a demonstration, not a tamper-proof log: arbitrary code with direct access to the instance dictionary can still edit values or log records.

## Problem 17 — Control a threaded cache race

**Challenge.** Implement a cached non-data descriptor that computes at most once for simultaneous accesses on the same instance. Test with multiple threads and a shared deterministic counter. Explain the lock granularity trade-off.

**Design hints.** Use a lock, check the instance dictionary *again inside the lock*, and assign only after computation succeeds. Avoid assuming a check outside the lock alone is sufficient.

### Solution 17

**Reference implementation and executable tests**

In [18]:
class LockedCached:
    def __init__(self, function):
        self.function = function
        self.lock = Lock()

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        # The first cache lookup is normally bypassed after successful caching,
        # because the instance dictionary shadows this non-data descriptor.
        with self.lock:
            if self.name not in instance.__dict__:
                value = self.function(instance)
                instance.__dict__[self.name] = value
            return instance.__dict__[self.name]


class ConcurrentCalculation:
    def __init__(self):
        self.calls = 0

    @LockedCached
    def expensive(self):
        self.calls += 1
        return 12345


concurrent = ConcurrentCalculation()
results = []
threads = [Thread(target=lambda: results.append(concurrent.expensive)) for _ in range(16)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
assert len(results) == 16
assert results == [12345] * 16
assert concurrent.calls == 1
print("Problem 17 passed: one computation for concurrent first reads.")

Problem 17 passed: one computation for concurrent first reads.


**Why it works / design review.** Double-checked caching under a lock prevents duplicate computation within this descriptor, assuming clients do not concurrently manipulate the same attribute directly. The descriptor-level lock is shared across instances, so it can serialize unrelated workloads; per-instance locks or `functools.cached_property` plus explicit synchronization may suit different needs.

## Problem 18 — Combine descriptors with dataclasses safely

**Challenge.** Use a dataclass for ordinary fields and an unannotated descriptor for a validated `age` attribute. Initialize the descriptor through an `InitVar` and `__post_init__`. Verify which attributes dataclass field introspection includes.

**Design hints.** Dataclasses inspect class attributes to decide defaults. Keep this example unambiguous: the descriptor is *not* a dataclass field, and an `InitVar` carries its constructor input.

### Solution 18

**Reference implementation and executable tests**

In [19]:
@dataclass
class RegisteredPerson:
    name: str
    initial_age: InitVar[int]
    age = BoundedInt(0, 130)  # deliberately NOT a dataclass-annotated field

    def __post_init__(self, initial_age):
        self.age = initial_age


person = RegisteredPerson("Grace", 37)
assert person.name == "Grace" and person.age == 37
assert [field.name for field in fields(RegisteredPerson)] == ["name"]
expect_raises(ValueError, lambda: setattr(person, "age", 131))
assert person.age == 37
assert isinstance(RegisteredPerson.age, BoundedInt)
print("Problem 18 passed: explicit dataclass/descriptor integration.")

Problem 18 passed: explicit dataclass/descriptor integration.


**Why it works / design review.** `dataclasses.fields()` and `dataclasses.asdict()` will not include the descriptor-managed age field in this arrangement. If age must participate in serialization, equality, or representation, explicitly implement those requirements or design a dataclass-specific descriptor field with care.

## Problem 19 — Build a schema-driven model with transactional updates

**Challenge.** Construct a descriptor-based model with inherited field registration, normalized values, unknown-field rejection, complete constructor validation, `to_dict()`, and atomic multi-field `update()`. Use integer cents for prices instead of inexact floating-point money.

**Design hints.** A descriptor owns *one-field* normalization; the model validates *all* proposed changes before committing them. Walk `reversed(cls.__mro__)` to collect inherited field descriptors in base-to-derived order. Store field metadata per descriptor, but field values per instance.

### Solution 19

**Reference implementation and executable tests**

In [20]:
def validate_sku(value):
    if not isinstance(value, str) or not value.strip():
        raise ValueError("sku must be a nonempty string")
    return value.strip().upper()


def validate_nonnegative_int(value):
    if type(value) is not int:
        raise TypeError("expected an exact integer (not bool)")
    if value < 0:
        raise ValueError("value must be nonnegative")
    return value


def validate_label(value):
    if not isinstance(value, str) or not value.strip():
        raise ValueError("label must be a nonempty string")
    return value.strip()


class SchemaField:
    def __init__(self, validator):
        self.validator = validator

    def __set_name__(self, owner, name):
        self.name, self.storage_key = name, f"_{name}"

    def normalize(self, value):
        return self.validator(value)

    def __get__(self, instance, owner):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.storage_key]
        except KeyError:
            raise AttributeError(f"{self.name} not initialized") from None

    def __set__(self, instance, value):
        instance.__dict__[self.storage_key] = self.normalize(value)


class SchemaModel:
    __fields__ = {}

    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        registry = {}
        for ancestor in reversed(cls.__mro__):
            for name, attribute in vars(ancestor).items():
                if isinstance(attribute, SchemaField):
                    registry[name] = attribute
        cls.__fields__ = registry

    def __init__(self, **values):
        expected = set(type(self).__fields__)
        if set(values) != expected:
            raise ValueError(
                f"missing={sorted(expected - set(values))}; "
                f"unknown={sorted(set(values) - expected)}"
            )
        self.update(**values)

    def update(self, **changes):
        registry = type(self).__fields__
        unknown = set(changes) - set(registry)
        if unknown:
            raise ValueError(f"unknown fields: {sorted(unknown)}")
        staged = {
            registry[name].storage_key: registry[name].normalize(value)
            for name, value in changes.items()
        }
        # This class intentionally centralizes validated writes in one commit.
        self.__dict__.update(staged)

    def to_dict(self):
        return {name: getattr(self, name) for name in type(self).__fields__}


class InventoryItem(SchemaModel):
    sku = SchemaField(validate_sku)
    price_cents = SchemaField(validate_nonnegative_int)
    quantity = SchemaField(validate_nonnegative_int)


class LabeledItem(InventoryItem):
    label = SchemaField(validate_label)


item = InventoryItem(sku="  ab-7 ", price_cents=1299, quantity=3)
assert item.to_dict() == {"sku": "AB-7", "price_cents": 1299, "quantity": 3}
assert list(LabeledItem.__fields__) == ["sku", "price_cents", "quantity", "label"]
previous_state = item.to_dict()
expect_raises(ValueError, lambda: item.update(quantity=4, price_cents=-1))
expect_raises(TypeError, lambda: item.update(quantity=True))
assert item.to_dict() == previous_state
item.update(quantity=4, price_cents=1599)
assert item.to_dict()["quantity"] == 4
assert json.loads(json.dumps(item.to_dict())) == item.to_dict()
expect_raises(ValueError, lambda: InventoryItem(sku="X", price_cents=10))
expect_raises(ValueError, lambda: item.update(not_a_field=1))
child = LabeledItem(sku=" y ", price_cents=100, quantity=0, label="  Gift  ")
assert child.to_dict()["label"] == "Gift"
print("Problem 19 passed: inheritance, normalization, serialization and atomic updates.")

Problem 19 passed: inheritance, normalization, serialization and atomic updates.


**Why it works / design review.** The model stages normalized values before using its private storage keys, preventing partial updates on validation errors. `to_dict()` returns Python values directly (and is JSON-safe for this exercise's strings and integers). This is an in-memory example; concurrent callers and external persistence require further transactional coordination.

## Problem 20 — Show why descriptors are not security or immutable-storage barriers

**Challenge.** Use a descriptor-controlled write-once attribute to show that `object.__setattr__` still invokes descriptor enforcement, but direct private dictionary mutation can bypass it. Explain the boundary of the guarantee.

**Design hints.** Python attribute hooks manage *normal access*. A cooperating caller can directly alter a mutable object dictionary, so distinguish API-level validation from adversarial tamper resistance.

### Solution 20

**Reference implementation and executable tests**

In [21]:
class TokenRecord:
    token = WriteOnce()

    def __init__(self, token):
        self.token = token


token_record = TokenRecord("issued")
expect_raises(AttributeError, lambda: object.__setattr__(token_record, "token", "changed"))
assert token_record.token == "issued"
# Explicit demonstration of the limitation; do not use in application logic.
token_record.__dict__["_token"] = "bypassed"
assert token_record.token == "bypassed"
print("Problem 20 passed: API enforcement is not a security boundary.")

Problem 20 passed: API enforcement is not a security boundary.


**Why it works / design review.** Descriptors enforce ordinary attribute operations; they do **not** turn arbitrary in-process Python objects into secure immutable records. Use access controls, database constraints, immutable data designs, or protected process boundaries when those guarantees are actually required.

## Advanced review: design checklist and common mistakes

| Concern | Recommended practice | Frequent mistake |
|---|---|---|
| Class-level access | Return `self` when `instance is None`, if introspection is desired | Trying to read instance fields while `instance is None` |
| Storage | Use an instance dictionary, dedicated slot, or considered external map | Keeping all owners' values in `self.value` on one shared descriptor |
| Data precedence | `__set__` **or** `__delete__` makes a data descriptor | Treating delete-only descriptors as non-data |
| `__set_name__` | Rely on it at class creation; call manually for post-creation injection | Assuming runtime `setattr(Owner, name, desc)` triggers it |
| Validation | Normalize/check fully, then commit | Mutating state before raising |
| Mutable defaults | Call a factory per instance | Sharing a list attached to the descriptor |
| Cross-field logic | Validate and commit at model level | Coupling unrelated descriptors through partial updates |
| Lazy caching | Cache per instance and define invalidation/failure policy | Caching all instance results on the descriptor |
| Weak references | Require weakrefable, hashable owners; check value→key retention | Assuming weak keys alone eliminate retention risks |
| Concurrency | Use a lock when at-most-once computation matters | Expecting a dictionary lookup to ensure exactly-once computation |
| Introspection | Inspect `vars(Owner)` or `inspect.getattr_static()` | Assuming `Owner.name` always exposes the raw descriptor |
| Security | Describe validation as an API contract | Claiming descriptor enforcement prevents direct memory mutation |

**Performance note:** descriptor-based abstractions add attribute-access overhead. Measure real workloads before replacing clear code with caches, custom lookup overrides, or metaclasses. Prefer existing standard-library utilities (e.g., `property`, `functools.cached_property`) where their exact semantics match your needs.

### Additional unsolved extension problems

1. Make a *per-instance-lock* cached descriptor with a slot-friendly synchronization strategy. Which object owns the lock, and when can it be reclaimed?
2. Modify `SchemaModel` so field overrides (including overriding a field with a **non-descriptor**) correctly remove inherited schema entries.
3. Add a JSON serializer for `datetime` and `Decimal` with explicit type conversion and versioned schema rules.
4. Create a validated observable list that prevents bypassing validation through `obj.labels.append(invalid_value)`.
5. Implement a weak-value cache whose cached results may be garbage-collected and transparently recomputed. Explain why not every return value can be weak-referenced.
6. Design a class that uses `__slots__` and multiple inheritance, documenting layout conflicts and descriptor storage ownership.

Each extension should include positive tests, negative tests, class-level introspection, independent-instance checks, and a concise statement of trade-offs.

In [22]:
# Final executable smoke checks across the completed problem set.
assert isinstance(EventLog.timestamp, UTCClock)
assert PrecedenceCase.data is vars(PrecedenceCase)["data"]
assert article.slug == "another-title"
assert result.score == 90
assert first_task.labels == ["docs"]
assert interval.start <= interval.end
assert report_a.calls == 2
assert service.calls == 2
assert slot_report.calls == 2
assert concurrent.calls == 1
assert child.to_dict()["sku"] == "Y"
assert token_record.token == "bypassed"  # intentional limitation demonstration
print("ALL 20 PROBLEM SOLUTIONS AND FINAL SMOKE CHECKS PASSED")

ALL 20 PROBLEM SOLUTIONS AND FINAL SMOKE CHECKS PASSED
